# Task 25 — Hate Speech Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sAndreotti/SpecialGems/blob/main/task_25_hate_speech/colab_train.ipynb)

Detect hate speech and toxic content in Italian social media posts.

| | |
|---|---|
| **Model** | `google/gemma-3-1b-it` |
| **Dataset** | `evalitahf/hatespeech_detection` |
| **Metrics** | Macro-F1 (EVALITA), HateCheck |
| **Framework** | [Unsloth](https://github.com/unslothai/unsloth) + TRL SFTTrainer |

> **Runtime:** select **GPU** in *Runtime → Change runtime type* (T4 is free; L4/A100 are faster).


In [ ]:
# Verify GPU is available
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU.')
print('GPU detected:', result.stdout.strip())


In [ ]:
# Install Unsloth (includes fp16 patch for Gemma 3) and evaluation libraries
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl>=0.8.6 peft>=0.10.0 transformers>=4.40.0 accelerate bitsandbytes \
               datasets evaluate rouge_score bert_score sacrebleu seqeval
print('Installation complete.')


In [ ]:
import os

REPO_URL = 'https://github.com/sAndreotti/SpecialGems.git'
REPO_DIR = '/content/SpecialGems'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())


## (Optional) HuggingFace Login

Gemma 3 is a gated model. Accept the licence at [huggingface.co/google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) then run the cell below with your HF token.


In [ ]:
# Uncomment and run if the model is gated
# from huggingface_hub import login
# login()   # paste your token when prompted


In [ ]:
# Download and prepare the dataset for Task 25
!python task_25_hate_speech/dataset_prep.py --save_path ./data/task_25


## Training

Adjust `--num_epochs` and `--batch_size` as needed for your GPU.

| GPU | Suggested settings |
|---|---|
| T4 (15 GB) | `--model_size 1b --batch_size 4 --num_epochs 3` |
| L4 (22 GB) | `--model_size 4b --batch_size 4 --num_epochs 3` |
| A100 (40 GB) | `--model_size 4b --batch_size 8 --num_epochs 5` |


In [ ]:
!python task_25_hate_speech/train.py \
    --model_size 1b \
    --dataset evalitahf/hatespeech_detection \
    --output_dir ./checkpoints/task_25 \
    --num_epochs 3 \
    --batch_size 4


## (Optional) Push fine-tuned model to HuggingFace Hub


In [ ]:
# Uncomment to push to HuggingFace Hub
# from huggingface_hub import login
# login()
# MODEL_PATH = './checkpoints/task_25/final_model'
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# model.push_to_hub('your-username/gemma-3-task25-hate-speech-detection')
# tokenizer.push_to_hub('your-username/gemma-3-task25-hate-speech-detection')


## Evaluation

Open `task_25_hate_speech/evaluate.ipynb` for the full base vs. fine-tuned comparison.

Or run it directly:


In [ ]:
!pip install -q jupyter nbconvert
!jupyter nbconvert --to notebook --execute task_25_hate_speech/evaluate.ipynb \
    --output task_25_hate_speech/evaluate_output.ipynb
print('Evaluation complete. Open evaluate_output.ipynb to see results.')


## Tag this task as done on GitHub

Once satisfied with the results, tag the commit and trigger the GitHub Release workflow:

```bash
git tag task-25-done -m 'Task 25 complete'
git push origin --tags
```

You can also notify via GitHub Actions dispatch (see README for the full snippet).
